# Week 7: Search Query Intent Classification

This notebook builds and evaluates a model that classifies real estate search queries into three intent categories:

- Browsing
- Researching
- High-intent inquiry

The model uses TF-IDF text features and logistic regression. Performance is evaluated on a held-out test set, and predictions include confidence scores.

In [21]:
import sys
sys.path.append("..")

import pandas as pd

from scripts.query_intent_classifier import QueryIntentClassifier

In [22]:
df = pd.read_csv(
    "../data/processed/week7_intent_queries.csv"
)

df.head()

,query,intent
0,show me homes in San Diego,browsing
1,houses in La Jolla,browsing
2,fancy condos near the beach,browsing
3,homes with fireplaces,browsing
4,houses near downtown San Jose,browsing


In [23]:
df["intent"].value_counts()

intent
browsing               68
researching            67
high_intent_inquiry    67
Name: count, dtype: int64

## Train-Test Split and Model Training

The labeled query dataset is split into training and testing sets. The training set is used to train the TF-IDF logistic regression classifier, while the held-out test set is used to evaluate how well the model generalizes to unseen queries.

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df["query"],
    df["intent"],
    test_size=0.2,
    random_state=42,
    stratify=df["intent"]
)

print("Training queries:", len(X_train))
print("Testing queries:", len(X_test))

Training queries: 161
Testing queries: 41


In [25]:
classifier = QueryIntentClassifier()

classifier.train(
    X_train,
    y_train
)

In [26]:
y_pred = [
    classifier.predict(query)[0]
    for query in X_test
]

accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy: {accuracy:.2%}")

Test accuracy: 90.24%


In [27]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

                     precision    recall  f1-score   support

           browsing       0.78      1.00      0.88        14
high_intent_inquiry       1.00      0.93      0.96        14
        researching       1.00      0.77      0.87        13

           accuracy                           0.90        41
          macro avg       0.93      0.90      0.90        41
       weighted avg       0.92      0.90      0.90        41



## Confidence Scores

The classifier also returns a confidence score representing the model's estimated probability for the predicted intent.

In [29]:
test_queries = [
    "show me homes in San Jose",
    "compare home prices in Cupertino and Santa Clara",
    "find homes with pool under 1.5m with open houses this weekend"
]

for query in test_queries:
    intent, confidence = classifier.predict(query)

    print("Query:", query)
    print("Predicted intent:", intent)
    print(f"Confidence: {confidence:.2%}")
    print()

Query: show me homes in San Jose
Predicted intent: high_intent_inquiry
Confidence: 59.09%

Query: compare home prices in Cupertino and Santa Clara
Predicted intent: researching
Confidence: 67.47%

Query: find homes with pool under 1.5m with open houses this weekend
Predicted intent: high_intent_inquiry
Confidence: 78.86%



## Integration with Week 4 Query Parser

The intent classifier is combined with the Week 4 query parser so that each search query can return both structured search filters and a predicted language-based intent.

In [30]:
from scripts.query_parser import QueryParser

In [32]:
query_parser = QueryParser()

query = "find homes with pool under 1.5m with open houses this weekend"

parsed_filters = query_parser.parse(query)
intent, confidence = classifier.predict(query)

print("Query:", query)
print("Parsed filters:", parsed_filters)
print("Predicted intent:", intent)
print(f"Confidence: {confidence:.2%}")

Query: find homes with pool under 1.5m with open houses this weekend
Parsed filters: {'price_max': 1500000, 'amenities': ['pool']}
Predicted intent: high_intent_inquiry
Confidence: 78.86%


## Limitations

Intent predictions are based only on the wording of the search query. The model does not use user behavior, browsing history, CRM data, or actual purchase activity. Because short queries may fit more than one intent category, some predictions may be uncertain or even wrong. Confidence scores help indicate when the model is less certain about a prediction.

The classifier is also trained on a relatively small labeled dataset, so confidence may be lower for queries that use wording or search patterns that were not included in the training data.

## Evaluation Results

Browsing: precision 0.78, recall 1.00, F1 0.88. This means the model caught all browsing queries, but some queries predicted as browsing were actually from another class.

High-intent inquiry: precision 1.00, recall 0.93, F1 0.96. This means when the model predicted high-intent, it was always correct, and it found almost all of the actual high-intent queries.

Researching: precision 1.00, recall 0.77, F1 0.87. This means when the model predicted researching, it was always correct, but it missed some actual researching queries and classified them as another category.


The intent classifier achieved **90.24% accuracy** on the held-out test set. The classifier also returns confidence scores and was successfully integrated with the Week 4 query parser to provide both structured filters and predicted search intent.